This function contains the edge detection logic used in auto_estimate_chirp. It can be used to optimize the parameters for your system.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.ndimage import uniform_filter1d

def debug_trace_analysis(dataset, i=0, edge_width_fs=500):
    A = dataset.data
    t = A.time.to_numpy()
    # Extract the trace for the requested wavelength index
    trace = A.to_numpy()[:, i]
    wv_val = A.spectral.to_numpy()[i]

    # 1. Smoothing
    smoothed = savgol_filter(trace, 11, 2)
    
    # 2. Gradient (Raw Slope)
    grad = np.gradient(smoothed)
    
    # 3. Sustained Gradient (Moving Average)
    # Automatically calculate dt based on your time array (assumes t is in ps)
    dt = np.median(np.diff(t)) 
    window_size = max(1, int((edge_width_fs / 1000.0) / dt))
    sustained_grad = uniform_filter1d(grad, size=window_size)
    
    # 4. Find the peak of the edge (steepest part)
    idx_max = np.argmax(np.abs(sustained_grad))
    t0_found = t[idx_max]

    # 5. NEW: Find the start of the edge by backtracking
    # We walk backwards from idx_max until the raw gradient drops below 10% 
    # of its peak value, or changes sign (crosses zero).
    peak_grad_mag = np.abs(grad[idx_max])
    threshold_grad = 0.10 * peak_grad_mag  # 10% threshold
    
    idx_start = idx_max
    while idx_start > 0 and np.abs(sustained_grad[idx_start]) > threshold_grad:
        # Stop early if the slope changes direction (we hit the pre-pulse baseline)
        if grad[idx_start] * grad[idx_max] < 0:
            break
        idx_start -= 1
        
    t_start = t[idx_start]

    # --- Plotting ---
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

    # Top Plot: Data
    ax1.plot(t, trace, label='Raw Data', alpha=0.3, color='gray')
    ax1.plot(t, smoothed, label='SavGol Smoothed', color='blue', lw=2)
    
    # Plot both the start and the peak
    ax1.axvline(t_start, color='orange', linestyle='--', label=f'Edge Start: {t_start:.3f} ps')
    ax1.axvline(t0_found, color='red', linestyle='--', label=f'Edge Peak: {t0_found:.3f} ps')
    
    ax1.set_ylabel('$\Delta$A Signal')
    ax1.set_title(f'Trace Analysis for $\lambda$ = {wv_val:.1f} nm (Index {i})')
    ax1.legend()

    # Bottom Plot: Derivatives
    ax2.plot(t, grad, label='Raw Gradient', alpha=0.4, color='green')
    ax2.plot(t, sustained_grad, label=f'Sustained Gradient ({edge_width_fs}fs)', color='darkgreen', lw=2)
    
    ax2.axvline(t_start, color='orange', linestyle='--')
    ax2.axvline(t0_found, color='red', linestyle='--')
    
    # Add a horizontal line to show the threshold we used to find the start
    if grad[idx_max] > 0:
        ax2.axhline(threshold_grad, color='orange', linestyle=':', alpha=0.5, label='10% Threshold')
    else:
        ax2.axhline(-threshold_grad, color='orange', linestyle=':', alpha=0.5, label='10% Threshold')

    ax2.set_ylabel('Gradient Magnitude')
    ax2.set_xlabel('Time (ps)')
    ax2.legend()

    plt.tight_layout()
    plt.xlim(-1, 10)
    plt.show()

# Run it for the first wavelength
debug_trace_analysis(dataset, i=0)

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d

def debug_fft_edge_detection(dataset, i=0, t_range=(-1, 8), cutoff_freq=2.0, edge_width_fs=200):
    """
    Denoises a trace using a Low-Pass FFT Filter, then finds the START 
    of the rising/falling edge using moving average backtracking.
    """
    A = dataset.data
    full_t = A.time.to_numpy()
    full_trace = A.to_numpy()[:, i]
    wv_val = A.spectral.to_numpy()[i]

    # 1. Slice the time and data arrays
    t_mask = (full_t >= t_range[0]) & (full_t <= t_range[1])
    t = full_t[t_mask]
    trace = full_trace[t_mask]

    # 2. Compute FFT and Frequencies
    n = len(trace)
    dt = np.median(np.diff(t))
    
    fhat = np.fft.fft(trace, n)                 
    psd = fhat * np.conj(fhat) / n              
    freq = np.fft.fftfreq(n, d=dt)              
    psd_real = np.abs(psd)

    # 3. Apply Low-Pass Filter
    indices = np.abs(freq) <= cutoff_freq 
    fhat_clean = fhat * indices 
    trace_clean = np.fft.ifft(fhat_clean).real

    # 4. Find the Peak of the Edge (using the cleaned trace)
    window_size = max(1, int((edge_width_fs / 1000.0) / dt))
    grad_clean = np.gradient(trace_clean)
    sustained_grad = uniform_filter1d(grad_clean, size=window_size)
    
    idx_max = np.argmax(np.abs(sustained_grad))
    t0_found = t[idx_max]

    # 5. Find the Start of the Edge (Backtracking on cleaned trace)
    trace_clean_ma = uniform_filter1d(trace_clean, size=window_size)
    peak_val = np.abs(trace_clean_ma[idx_max])
    threshold_val = 0.10 * peak_val  # 10% threshold
    
    idx_start = idx_max
    while idx_start > 0 and np.abs(trace_clean_ma[idx_start]) > threshold_val:
        idx_start -= 1
        
    t_start = t[idx_start]

    # --- Plotting (3 Panels) ---
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 12))

    # Panel 1: Time Domain Signal
    ax1.plot(t, trace, color='gray', alpha=0.4, label='Raw Data (Sliced)')
    ax1.plot(t, trace_clean, color='blue', lw=2, label=f'FFT Denoised (< {cutoff_freq} THz)')
    ax1.plot(t, trace_clean_ma, color='purple', lw=2, linestyle='-.', label=f'Signal MA ({edge_width_fs}fs)')
    
    ax1.axvline(t_start, color='orange', linestyle='--', lw=2, label=f'Edge Start: {t_start:.3f} ps')
    ax1.axvline(t0_found, color='red', linestyle='--', lw=2, label=f'Edge Peak: {t0_found:.3f} ps')
    
    sign = np.sign(trace_clean_ma[idx_max]) if trace_clean_ma[idx_max] != 0 else 1
    ax1.axhline(sign * threshold_val, color='orange', linestyle=':', lw=2, label='10% MA Threshold')

    ax1.set_title(f'Time Domain Signal: $\lambda$ = {wv_val:.1f} nm')
    ax1.set_ylabel('$\Delta$A Signal')
    ax1.legend(loc='upper right')

    # Panel 2: Time Domain Gradients
    ax2.plot(t, grad_clean, color='green', alpha=0.5, label='FFT Cleaned Gradient')
    ax2.plot(t, sustained_grad, color='darkgreen', lw=2, label=f'Sustained Gradient ({edge_width_fs}fs)')
    
    ax2.axvline(t_start, color='orange', linestyle='--', lw=2)
    ax2.axvline(t0_found, color='red', linestyle='--', lw=2)
    
    ax2.set_title('Gradient Tracking')
    ax2.set_ylabel('Gradient Magnitude')
    ax2.legend(loc='upper right')

    # Panel 3: Frequency Domain (PSD)
    L = np.arange(1, int(np.floor(n/2)), dtype=int) 
    ax3.plot(freq[L], psd_real[L], color='indigo', label='Power Spectral Density')
    ax3.axvline(cutoff_freq, color='red', linestyle='--', lw=2, label=f'Cutoff: {cutoff_freq} THz')
    ax3.axvspan(cutoff_freq, np.max(freq[L]), color='red', alpha=0.1, label='Rejected Noise')

    ax3.set_title('Frequency Domain Filter')
    ax3.set_ylabel('Power')
    ax3.set_xlabel('Frequency (THz)')
    ax3.set_yscale('log') 
    ax3.legend(loc='upper right')

    plt.tight_layout()
    plt.show()

# Run it to test:
# debug_fft_edge_detection(dataset, i=1, t_range=(-1, 8), cutoff_freq=2.0, edge_width_fs=200)